In [ ]:
import json
import re
import pandas as pd
from pathlib import Path
from datetime import datetime
from dataclasses import dataclass
from typing import Callable

In [ ]:
WORKDIR = Path("./_workspace")
WORKDIR.mkdir(exist_ok=True)

In [ ]:
# z zadania 1

import functools
from datasets import load_dataset

@functools.lru_cache(maxsize=4)
def get_imdb_subset(split: str, n: int):
    ds = load_dataset("stanfordnlp/imdb", split=split).shuffle(seed=42).select(range(n))
    return [(r["text"], r["label"]) for r in ds]

samples = get_imdb_subset("train", 2000)

In [ ]:
#dane
samples_dq = get_imdb_subset("train", 2000)

df_pd = pd.DataFrame(samples_dq, columns=["text", "label"])
df_pd["word_count"] = df_pd["text"].str.split().str.len()
df_pd["char_count"] = df_pd["text"].str.len()


@dataclass
class Rule:
    name: str
    check: Callable
    severity: str = "warning"


class DataContract:
    def __init__(self, name: str):
        self.name = name
        self.rules: list[Rule] = []

    def add_rule(self, name: str, check: Callable, severity: str = "warning"):
        if severity not in {"info", "warning", "error"}:
            raise ValueError("severity musi być jednym z: info, warning, error")

        self.rules.append(Rule(name=name, check=check, severity=severity))


class DataValidator:
    def __init__(self, contract: DataContract):
        self.contract = contract

    def validate(self, df) -> dict:
        report = {
            "contract_name": self.contract.name,
            "timestamp": datetime.now().isoformat(),
            "rules": {}
        }

        failed_errors = []

        for rule in self.contract.rules:
            try:
                result = rule.check(df)

                if isinstance(result, tuple):
                    passed, details = result
                else:
                    passed = bool(result)
                    details = ""

            except Exception as e:
                passed = False
                details = f"Błąd podczas sprawdzania reguły: {e}"

            report["rules"][rule.name] = {
                "passed": bool(passed),
                "severity": rule.severity,
                "details": details
            }

            if not passed and rule.severity == "error":
                failed_errors.append(rule.name)

        if failed_errors:
            raise ValueError(f"Nie przeszły krytyczne reguły jakości danych: {failed_errors}")

        return report

In [ ]:
#reguły
contract = DataContract("IMDB data quality contract")

contract.add_rule(
    "no_nulls",
    lambda df: (
        df[["text", "label"]].isnull().sum().sum() == 0,
        f"Liczba NULL w text i label: {int(df[['text', 'label']].isnull().sum().sum())}"
    ),
    severity="error"
)

contract.add_rule(
    "labels_in_set",
    lambda df: (
        set(df["label"].unique()).issubset({0, 1}),
        f"Unikalne wartości label: {sorted(df['label'].unique().tolist())}"
    ),
    severity="error"
)

contract.add_rule(
    "min_word_count",
    lambda df: (
        (df["word_count"] >= 5).all(),
        f"Liczba recenzji krótszych niż 5 słów: {int((df['word_count'] < 5).sum())}"
    ),
    severity="error"
)

contract.add_rule(
    "max_word_count",
    lambda df: (
        (df["word_count"] <= 2000).all(),
        f"Liczba recenzji dłuższych niż 2000 słów: {int((df['word_count'] > 2000).sum())}"
    ),
    severity="error"
)

contract.add_rule(
    "no_duplicates",
    lambda df: (
        df["text"].duplicated().sum() == 0,
        f"Liczba duplikatów text: {int(df['text'].duplicated().sum())}"
    ),
    severity="error"
)

contract.add_rule(
    "class_balance",
    lambda df: (
        0.5 <= (df["label"].value_counts().min() / df["label"].value_counts().max()) <= 1.5,
        f"Stosunek klas: {df['label'].value_counts().min() / df['label'].value_counts().max():.3f}"
    ),
    severity="warning"
)

contract.add_rule(
    "no_html_tags",
    lambda df: (
        df["text"].str.contains(r"<[^>]+>", regex=True).sum() == 0,
        f"Liczba recenzji zawierających HTML: {int(df['text'].str.contains(r'<[^>]+>', regex=True).sum())}"
    ),
    severity="warning"
)

In [ ]:
#walidacja
validator = DataValidator(contract)

try:
    report = validator.validate(df_pd)
    print("Walidacja zakończona. Krytyczne reguły przeszły poprawnie.")
except ValueError as e:
    print("Walidacja nie przeszła:")
    print(e)
    report = None

Walidacja zakończona. Krytyczne reguły przeszły poprawnie.


In [ ]:
#zapis raportu
if report is not None:
    report_path = WORKDIR / "data_quality_report.json"

    with open(report_path, "w", encoding="utf-8") as f:
        json.dump(report, f, ensure_ascii=False, indent=2)

    print(f"Raport zapisany do: {report_path}")

    print("\nPodsumowanie reguł:")
    for rule_name, rule_result in report["rules"].items():
        status = "OK" if rule_result["passed"] else "NIE OK"
        print(f"{rule_name}: {status} ({rule_result['severity']}) - {rule_result['details']}")

Raport zapisany do: _workspace/data_quality_report.json

Podsumowanie reguł:
no_nulls: OK (error) - Liczba NULL w text i label: 0
labels_in_set: OK (error) - Unikalne wartości label: [0, 1]
min_word_count: OK (error) - Liczba recenzji krótszych niż 5 słów: 0
max_word_count: OK (error) - Liczba recenzji dłuższych niż 2000 słów: 0
no_duplicates: OK (error) - Liczba duplikatów text: 0
class_balance: OK (warning) - Stosunek klas: 1.000
no_html_tags: NIE OK (warning) - Liczba recenzji zawierających HTML: 1189


### Wnioski

Walidacja danych zakończyła się poprawnie, ponieważ wszystkie reguły krytyczne oznaczone jako `error` zostały spełnione. W danych nie było braków w kolumnach `text` i `label`, etykiety miały tylko dozwolone wartości 0 i 1, nie wykryto duplikatów oraz wszystkie recenzje mieściły się w przyjętych limitach długości.

Zbiór jest też dobrze zbalansowany — stosunek klas wyniósł 1.000, czyli liczba recenzji pozytywnych i negatywnych była taka sama. To ważne, ponieważ przy takim rozkładzie model uczący się na tych danych nie byłby od początku faworyzowany w stronę jednej klasy.

Jedyna niespełniona reguła to `no_html_tags`, ponieważ 1189 recenzji zawierało znaczniki HTML. Reguła miała poziom `warning`, więc nie zatrzymała walidacji, ale wskazuje problem, który warto naprawić przed dalszym przetwarzaniem danych. Przed analizą tekstu lub trenowaniem modelu należałoby usunąć znaczniki HTML, żeby nie wpływały na tokenizację i wyniki.